# Notebook for extracting and analyzing energies data

Almost all of the data are stored in various `data.csv` (see file tree below)

Each `data.csv` contains energies over variations in basis function, distance (if applicable), gamma, etc.

The `metadata.json` file contains metadata of all the various parameters used in the runs, 
for example, the list of basis sets, gammas, how the output files are named, and so on.


## Before running this notebook

### To generate `data.csv` (eg: cu):
```
<from systems folder>
mkdir cu
cd cu
mkdir method && cd method
<write metadata.json and some_input.inp> # -> best to copy existing and modify
cd ../../
python generate_inputs_and_folder.py cu/method/metadata.json
python run_inputs_and_folder.py cu/method/metadata.json
python tabulate_outputs_and_folders.py cu/method/metadata.json
```

In [ ]:
%%bash
## Locations of `data.csv`
tree -P 'data.csv' --prune
find . -name 'data.csv'

## Imports

In [ ]:
import pandas as pd
import importlib
import matplotlib as mpl
import matplotlib.pyplot as pp
from matplotlib.gridspec import GridSpec
from matplotlib.backends.backend_pdf import PdfPages
from matplotlib.lines import Line2D
import numpy as np
import scipy as sp
from ipywidgets import widgets
import os
from typing import Optional, List, Tuple
from functools import reduce
import datetime
import functions_for_analyze as ffa
dailies = os.path.realpath('../../dailies/')
%matplotlib widget

pd.set_option('display.max_colwidth', None)

# Functions 

## Daily reporters

In [ ]:
import _daily_reporters as dr

importlib.reload(dr)
dump_daily = dr.dump_daily
save_fig_as_pdf = dr.save_fig_as_pdf

## Data retrieval

In [ ]:
importlib.reload(ffa)
get_data =  ffa.get_data
assign_basis_size = ffa.assign_basis_size

## Comparing fragments to 9999.0 distance

In [ ]:
importlib.reload(ffa)
add_suffix =  ffa.add_suffix
compare_fragment_to_dimer = ffa.compare_fragment_to_dimer

## Basis set extrapolation

In [ ]:
importlib.reload(ffa)

get_cbs_from_pair = ffa.get_cbs_from_pair
get_cbs_from_df = ffa.get_cbs_from_df
plot_cbs_extrapolation = ffa.plot_cbs_extrapolation


# Results

## Results: optimal gamma

In [ ]:
def get_x_at_ymin(x, y):
    return x[np.argmin(y)]

get_x_at_ymin([0,1,2,3,4], [2,2,0,0,1])

### Ar

In [ ]:
data_file = 'ar_fully_correlated/default/data.csv'
data = get_data(data_file)
assign_basis_size(data)
#data = data.sort_values(by='basis_sizes')
basis_size_dict = {}
for n, basis in enumerate(data.bases.values):
    basis_size_dict[basis] = data.basis_sizes.values[n]

print(basis_size_dict)
print(data.columns)

print(set(data['ansatzes'].values))
print(set(data['core'].values))
print(set(data['bases'].values))
print(set(data['GEM_BETA'].values))

# ============================
filters = dict(
    ansatzes = ['3C(FIX,HY1)'],
    #core = ['core,1'],
    bases = ['aug-cc-pVTZ'],
)
cmap = pp.get_cmap('viridis')



bases = [bas for bas in set(data['bases'].values) if 'cc-pV' in bas]
bases = sorted(bases, key=lambda bas: basis_size_dict[bas])

n = len(bases)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 0.8, n)]

core = 'core,5'
fig,ax = pp.subplots()
fig.suptitle(f"Corr_ener vs gamma for Ar {core}")

filters['core'] = [core]
for n, basis in enumerate(bases):
    filters['bases'] = [basis]
    
    df = data.copy()
    for key, values in filters.items():
        df = df[df[key].isin(values)]

    ax.plot(df.GEM_BETA.values, df.CORR_ENER.values, color=colors[n], label=basis)
    ax.axvline(get_x_at_ymin(df.GEM_BETA.values, df.CORR_ENER.values), color=colors[n], linestyle='--')
    
ax.legend()

#### Differences between cores

In [ ]:

cores2comp = ['core,0', 'core,1', 'core,5']
filters['core'] = cores2comp

fig = pp.figure()
NGx = 3
NGy = 2
gs = pp.GridSpec(
    NGx, NGy, figure=fig,
    hspace=0.5
)

axes_total = []
axes_diff = []

sharex = None
for n in range(3):
    if n > 0:
        sharex = axes_total[n-1]
        
    ax = fig.add_subplot(
            gs[n * (NGx//3) : (n+1) * (NGx//3), : NGy//2 ] , sharex=sharex
        )
    axes_total.append(ax)
    if n == 2:
        break
    pp.setp(axes_total[n].get_xticklabels(), visible=False)


sharex = None
for n in range(3):
    if n > 0:
        sharex = axes_diff[n-1]
        
    ax = fig.add_subplot(
            gs[n * (NGx//3) : (n+1) * (NGx//3), NGy//2 :  ] , sharex=sharex
        )
    axes_diff.append(ax)
    if n < 2:
        pp.setp(axes_diff[n].get_xticklabels(), visible=False)
    
#axl = axes_diff[-1] 
#axl.set_axis_off()

fig.suptitle('Ar correner vs gamma at various core approximations')

opt_gammas = {
    'bases' : [], 'core,0' : [], 'core,1' : [], 'core,5': [], 
    'diff(0-1)' : [], 'diff(0-5)' : [], 'diff(1-5)' : []
}


for n, basis in enumerate(bases):
    filters['bases'] = [basis]
    
    df = data.copy()
    for key, values in filters.items():
        df = df[df[key].isin(values)]

    df0 = df[df['core']==cores2comp[0]]
    df1 = df[df['core']==cores2comp[1]]
    df5 = df[df['core']==cores2comp[2]]



    gammas = df0.GEM_BETA.values
    #========================================
    ener_diff01 = df0.CORR_ENER.values - df1.CORR_ENER.values
    ax = axes_diff[0]
    ax.plot(gammas, ener_diff01, color=colors[n], label=basis)
    ax.set_title('Difference (0-1)')
   
    opt_gamma = get_x_at_ymin(gammas, ener_diff01)
    opt_gammas['bases'].append(basis)
    opt_gammas['diff(0-1)'].append(opt_gamma)
    ax.axvline(opt_gamma, color=colors[n], linestyle='--')
    #========================================
    ener_diff05 = df0.CORR_ENER.values - df5.CORR_ENER.values
    ax = axes_diff[1]
    ax.plot(gammas, ener_diff05, color=colors[n], label=basis)
    ax.set_title('Difference (0-5)')
   
    opt_gamma = get_x_at_ymin(gammas, ener_diff05)
    #opt_gammas['bases'].append(basis)
    opt_gammas['diff(0-5)'].append(opt_gamma)
    ax.axvline(opt_gamma, color=colors[n], linestyle='--')
    #========================================
    ener_diff15 = df1.CORR_ENER.values - df5.CORR_ENER.values
    ax = axes_diff[2]
    ax.plot(gammas, ener_diff15, color=colors[n], label=basis)
    ax.set_title('Difference (1-5)')
   
    opt_gamma = get_x_at_ymin(gammas, ener_diff15)
    #opt_gammas['bases'].append(basis)
    opt_gammas['diff(1-5)'].append(opt_gamma)
    ax.axvline(opt_gamma, color=colors[n], linestyle='--')
    
    for i, core in enumerate(cores2comp):
        ener = [df0, df1, df5][i].CORR_ENER.values
        og = get_x_at_ymin(gammas, ener)
        opt_gammas[core].append(
            og)
        axes_total[i].plot(
            gammas, ener, color=colors[n], label=basis
        )
        axes_total[i].set_title(core)
        axes_total[i].axvline(og, color=colors[n], linestyle='--')
    

#axl.legend(
#    *ax.get_legend_handles_labels()
#)

In [ ]:
df = pd.DataFrame(opt_gammas)
display(df)

In [ ]:
dump_daily(df, fname='ar_opt_gammas.csv')

In [ ]:
dump_daily(fig, fname = f'ar_corr_ener_vs_gamma.pdf', note='Ar fully correlated gamma optimization')

### Ne

In [ ]:
data_file = 'ne_fully_correlated/default/data.csv'
data = get_data(data_file)
assign_basis_size(data)
#data = data.sort_values(by='basis_sizes')
basis_size_dict = {}
for n, basis in enumerate(data.bases.values):
    basis_size_dict[basis] = data.basis_sizes.values[n]

print(basis_size_dict)
print(data.columns)

print(set(data['ansatzes'].values))
print(set(data['core'].values))
print(set(data['bases'].values))
print(set(data['GEM_BETA'].values))

# ============================
filters = dict(
    ansatzes = ['3C(FIX,HY1)'],
    core = ['core,1'],
    bases = ['aug-cc-pVTZ'],
)
cmap = pp.get_cmap('viridis')



bases = [bas for bas in set(data['bases'].values) if 'cc-pV' in bas]
bases = sorted(bases, key=lambda bas: basis_size_dict[bas])

n = len(bases)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 0.8, n)]

core = 'core,1'
filters['core'] = [core]
fig,ax = pp.subplots()
fig.suptitle(f"Corr_ener vs gamma for Ne {core}")
for n, basis in enumerate(bases):
    filters['bases'] = [basis]

    df = data.copy()
    for key, values in filters.items():
        df = df[df[key].isin(values)]

    ax.plot(df.GEM_BETA.values, df.CORR_ENER.values, color=colors[n], label=basis)
    ax.axvline(get_x_at_ymin(df.GEM_BETA.values, df.CORR_ENER.values), color=colors[n], linestyle='--')
    
ax.legend()


In [ ]:

dr.dump_daily(fig, fname=f'ne_{core}_ener_vs_gamma.pdf', note=f'Correlation energy vs gamma for various bases for Ne with {core}')

#### Now the difference between core,1 and core,0

In [ ]:

cores2comp = ['core,0', 'core,1']
filters['core'] = cores2comp

fig = pp.figure()
NG = 3
gs = pp.GridSpec(
    NG, NG, figure=fig,
    hspace=0.5
)

axes = []
for n in range(3):
    axes.append(
        fig.add_subplot(
            gs[n * (NG//3) : (n+1) * (NG//3), : NG//3 * 2 ] 
        )
    )
axl =fig.add_subplot(gs[:, NG//3 * 2:]) 
axl.set_axis_off()

fig.suptitle('Ne correlation energies at various core approximations')

opt_gammas = {'bases' : [], 'core,0' : [], 'core,1' : [], 'diff' : []}

for n, basis in enumerate(bases):
    filters['bases'] = [basis]
    
    df = data.copy()
    for key, values in filters.items():
        df = df[df[key].isin(values)]

    df0 = df[df['core']==cores2comp[0]]
    df1 = df[df['core']==cores2comp[1]]




    gammas = df0.GEM_BETA.values
    ener_diff = df0.CORR_ENER.values - df1.CORR_ENER.values
    ax = axes[-1]
    ax.plot(gammas, ener_diff, color=colors[n], label=basis)
    ax.set_title('Difference ')
   
    opt_gamma = get_x_at_ymin(gammas, ener_diff)
    opt_gammas['bases'].append(basis)
    opt_gammas['diff'].append(opt_gamma)
    ax.axvline(opt_gamma, color=colors[n], linestyle='--')
    
    for i, core in enumerate(cores2comp):
        ener = [df0, df1][i].CORR_ENER.values
        og = get_x_at_ymin(gammas, ener)
        opt_gammas[core].append(
            og)
        axes[i].plot(
            gammas, ener, color=colors[n], label=basis
        )
        axes[i].set_title(core)
        axes[i].axvline(og, color=colors[n], linestyle='--')
    

axl.legend(
    *ax.get_legend_handles_labels()
)

In [ ]:
dr.dump_daily(fig, fname=f'ne_ener_vs_gamma.pdf', note=f'Correlation energy vs gamma for various bases for Ne')

In [ ]:
df = pd.DataFrame(opt_gammas)


In [ ]:
dr.dump_daily(df, fname=f'ne_opt_gammas.csv')

## Results: BSE of nonXG F12

### AR
```
 	bases 	core,0 	core,1 	core,5 	diff(0-1) 	diff(0-5) 	diff(1-5)
0 	aug-cc-pVDZ 	2.6 	2.2 	1.2 	4.0 	3.0 	2.4
1 	aug-cc-pVTZ 	3.0 	2.4 	1.6 	4.0 	3.0 	2.6
2 	aug-cc-pVQZ 	3.0 	2.6 	2.0 	4.0 	3.0 	2.6
3 	aug-cc-pV5Z 	3.5 	2.6 	2.0 	4.0 	3.5 	3.0

```

#### Reference data from DFMP2

In [ ]:
system = "ar_fully_correlated//"
method = "dfmp2"
ref_data = get_data(f"{system}/{method}/data.csv")

assign_basis_size(ref_data)
rdf0 = ref_data[ref_data.core == 'core,0']
rdf1 = ref_data[ref_data.core == 'core,1']
rdf5 = ref_data[ref_data.core == 'core,5']

In [ ]:
data_file = 'ar_fully_correlated/default/data.csv'
data = get_data(data_file)
assign_basis_size(data)
#data = data.sort_values(by='basis_sizes')
basis_size_dict = {}
for n, basis in enumerate(data.bases.values):
    basis_size_dict[basis] = data.basis_sizes.values[n]

#set(data.GEM_BETA.values)

#### Compare BSE of F12 vs DFMP2

In [ ]:

ref_filters = dict(
    bases = [bas for bas in data.bases if 'cc-pw' in bas]
)

filters = dict(
    ansatzes = ['3C(FIX,HY1)'],
    bases = [bas for bas in data.bases if 'cc-pV' in bas]
    #core = ['core,1'],
    #GEM_BETA = ['aug-cc-pVTZ'],
)


gammas = [1.6, 2.6, 3.0]
cores2comp = ['core,5',  'core,1', 'core,0']



fig = pp.figure()
ncols = 3
nrows = 3


NGx = 3
NGy = 24
leg_width = 7
gs = pp.GridSpec(
    NGx, NGy+leg_width, figure=fig,
    hspace=0.05
)

cmap = pp.get_cmap('Blues_r')
ref_cmap = pp.get_cmap('Reds_r')
#n = len(bases)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 0.8, nrows)]
ref_colors = [ref_cmap(i) for i in np.linspace(0, 0.8, nrows)]

fig.suptitle('Ar correner BSE at various core approximations and gammas')

axes = [[] for _ in range(NGx)]

sharex = None
exp = 3
for n in range(nrows):

    filters['core'] = [cores2comp[n]]
    ref_filters['core'] = [cores2comp[n]]
    if n > 0:
        sharex = axes[n-1][m]

    sharey=None
    for m in range(ncols):
        if m > 0:
            sharey = axes[n][m - 1]
            
        ax = fig.add_subplot(
            gs[
                n * (NGx//nrows) : (n+1) * (NGx//nrows), 
                m * (NGy//(ncols)) : (m+1) * (NGy//(ncols)) 
            ], 
            sharex=sharex, sharey=sharey
            )
        axes[n].append(ax)

        if m > 0:
            pp.setp(axes[n][m].get_yticklabels(), visible = False)
        if n < NGx - 1:    
            pp.setp(axes[n][m].get_xticklabels(), visible=False)

        #for i, gamma in enumerate(gammas):
        
        gamma = filters['GEM_BETA'] = [gammas[m]]

        #===================
        df = data.copy()
        for key, values in filters.items():
            df = df[df[key].isin(values)]

        #display(df)
        plot_cbs_extrapolation(
            df.basis_sizes.values, df.CORR_ENER.values,
            ax = ax, label=f'gamma={gamma}', marker='o', color=colors[1],
            exp=exp
                      )
        #====================
        df = ref_data.copy()
        for key, values in ref_filters.items():
            df = df[df[key].isin(values)]

        plot_cbs_extrapolation(
            df.basis_sizes.values, df.CORR_ENER.values,
            ax = ax, label=f'gamma={gamma}', marker='x', color=ref_colors[1],
            exp=exp
                      )
        
for n in range(nrows):
    axes[0][n].set_title(f"gamma={gammas[n]}")
    axes[-1][n].set_xlabel(f"$1/X^{exp}$")

for n in range(ncols):
    axes[n][0].set_ylabel(f"{cores2comp[n]}")
            
axl = fig.add_subplot(
    gs[:, NGy:], 
    )

custom_lines = [Line2D([0], [0], color=colors[1], lw=4),
                Line2D([0], [0], color=ref_colors[1], lw=4),
               ]
axl.set_axis_off()
axl.legend(custom_lines, ("F12", "dfmp2"))

axl.text(0.5, 0.5, "basis dfmp2:\n aug-cc-pwCVXZ", horizontalalignment='center',
     verticalalignment='center', transform=axl.transAxes, wrap=True)
axl.text(0.5, 0.7, "basis f12:\n aug-cc-pVXZ", horizontalalignment='center',
     verticalalignment='center', transform=axl.transAxes, wrap=True)

In [ ]:
dr.dump_daily(fig, fname='ar_ce_bse_dfmp2_pwCVX_vs_f12_pVXZ.pdf', note='self explanatory')

#### Differences between cores
```
	bases	core,0	core,1	core,5	diff(0-1)	diff(0-5)	diff(1-5)
0	aug-cc-pVDZ	2.6	2.2	1.2	4.0	3.0	2.4
1	aug-cc-pVTZ	3.0	2.4	1.6	4.0	3.0	2.6
2	aug-cc-pVQZ	3.0	2.6	2.0	4.0	3.0	2.6
3	aug-cc-pV5Z	3.5	2.6	2.0	4.0	3.5	3.0
```

In [ ]:
ref_filters = dict(
    bases = [bas for bas in data.bases if 'cc-pw' in bas]
)

filters = dict(
    ansatzes = ['3C(FIX,HY1)'],
    bases = [bas for bas in data.bases if 'cc-pV' in bas]
    #core = ['core,1'],
    #GEM_BETA = ['aug-cc-pVTZ'],
)
cmap = pp.get_cmap('viridis')

gammas = [1.6, 2.6, 3.0]
diff2comp = [
    ('core,0', 'core,1'), 
    ('core,0', 'core,5'),
    ('core,1', 'core,5')
]



fig = pp.figure()
NGx = 3
NGy = 3
gs = pp.GridSpec(
    NGx, NGy, figure=fig,
    hspace=0.5
)

#colors = [cmap(i) for i in np.linspace(0, 0.8, n)]

fig.suptitle('Ar difference of correners, including BSE ')

axes = [[] for _ in range(NGx)]

sharex = None
for n in range(NGx):
    if n > 0:
        sharex = axes[n-1][m]

    sharey=None
    for m in range(NGy):
        if m > 0:
            sharey = axes[n][m - 1]
            
        ax = fig.add_subplot(
                gs[n * (NGx//3) : (n+1) * (NGx//3), m * (NGy//3) : (m+1) * (NGy//3) ] , sharex=sharex, sharey=sharey
            )
        axes[n].append(ax)

        if m > 0:
            pp.setp(axes[n][m].get_yticklabels(), visible = False)
        if n < NGx - 1:    
            pp.setp(axes[n][m].get_xticklabels(), visible=False)

        #for i, gamma in enumerate(gammas):
        
        filters['GEM_BETA'] = [gammas[m]]
        
        filters['core'] = [diff2comp[n][0]]
        ref_filters['core'] = [diff2comp[n][0]]
        df0 = data.copy()
        for key, values in filters.items():
            df0 = df0[df0[key].isin(values)]
        #====================
        rdf0 = ref_data.copy()
        for key, values in ref_filters.items():
            rdf0 = rdf0[rdf0[key].isin(values)]

        filters['core'] = [diff2comp[n][1]]
        ref_filters['core'] = [diff2comp[n][1]]
        df1 = data.copy()
        for key, values in filters.items():
            df1 = df1[df1[key].isin(values)]
        #====================
        rdf1 = ref_data.copy()
        for key, values in ref_filters.items():
            rdf1 = rdf1[rdf1[key].isin(values)]

        
        #display(df)
        ener_diff = df0.CORR_ENER.values - df1.CORR_ENER.values
        ref_ener_diff = rdf0.CORR_ENER.values - rdf1.CORR_ENER.values

        basis_sizes = df0.basis_sizes.values

        BSE_ener0 = get_cbs_from_pair(basis_sizes[-2:], df0.CORR_ENER.values[-2:])
        BSE_ener1 = get_cbs_from_pair(basis_sizes[-2:], df1.CORR_ENER.values[-2:])

        ref_BSE_ener0 = get_cbs_from_pair(basis_sizes[-2:], rdf0.CORR_ENER.values[-2:])
        ref_BSE_ener1 = get_cbs_from_pair(basis_sizes[-2:], rdf1.CORR_ENER.values[-2:])
        
        ener_diff = np.append(ener_diff, [BSE_ener0 - BSE_ener1])
        ref_ener_diff = np.append(ref_ener_diff, [ref_BSE_ener0 - ref_BSE_ener1])

        x = 1/basis_sizes**(exp)
        x = np.append(x, [0])
        #display(x, ref_ener_diff)
        ax.plot(x, ref_ener_diff, marker = 'x', color=ref_colors[1])
        ax.plot(x, ener_diff, label=f'gamma={gamma}', marker='o', color=colors[1])

for n in range(NGx):
    axes[0][n].set_title(f"gamma={gammas[n]}")

for n in range(NGy):
    axes[n][0].set_ylabel(f"{diff2comp[n]}")


In [ ]:
dr.dump_daily(fig, fname='Ar_diff_cor_ener_pwCVXZ_vs_pVXZ.pdf', note='difference of corr ener at various core approximations')

## Results: XG BSE of various core/MAFs

### AR

In [ ]:
system = "ar_fully_correlated//"
method = "xg"
data = get_data(f"{system}/{method}/data.csv")
assign_basis_size(data)

In [ ]:
for col in data.columns:
    display(col)
set(data.gamma_set)

#### Get a sense of "best" gammas

In [ ]:
fig = pp.figure()
gs = pp.GridSpec(1, 7)

ax = fig.add_subplot(gs[:, 0:3])
axl =fig.add_subplot(gs[:, 3:]) 


linestyles = ['solid', 'dotted', 'dashdot']
cmap = pp.get_cmap('Blues')
#gamma_sets =  set(data.gamma_set.values)
gamma_sets = [
    "1.20_1.20_1.20",
    "3.00_3.00_3.00",
    "1.20_3.00_1.70",
    "1.20_3.00_2.10",
    "1.20_3.00_2.50",
    "1.20_3.00_3.00",
]
n = len(gamma_sets)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0.2, 1, n)]

filters = dict(

    bases = [bas for bas in data.bases if 'cc-pV' in bas],
    xg_maf_folders = ['MAF_CORE_5'],
    #'ansatzes' : '3C(FIX,NOZ)',
    core = ['core,5'],
)
ref_filters = dict(
    bases = [bas for bas in ref_data.bases if 'cc-pw' in bas],
    core = ['core,5'],
)
for n, gamma in enumerate(gamma_sets):
    
    filters['gamma_set'] = [gamma]
    
    dff12 = data.copy()
    for key, val in filters.items():
        dff12 = dff12[dff12[key].isin(val)]
        
    #display(dff12)
    exp = 3
    line, ax = plot_cbs_extrapolation(dff12.basis_sizes.values, dff12['correlation energy'].values,
                           ax = ax, label=f'gamma={gamma}', marker='o', color=colors[n], exp=exp
                          )

dfmp2 = ref_data.copy()
for key, val in ref_filters.items():
    dfmp2 = dfmp2[dfmp2[key].isin(val)]

line, ax = plot_cbs_extrapolation(dfmp2.basis_sizes.values, dfmp2['CORR_ENER'].values,
                           ax = ax, label="dfmp2,pwC", marker='o', color=ref_colors[1], exp=exp
                          )

    
axl.set_axis_off()
axl.legend(*ax.get_legend_handles_labels())
fig.suptitle('BSE of Ar (fully correlated) in F12XG')

In [ ]:
dr.dump_daily(fig, fname="bse_of_fully_correlated_ar_xg_pV_vs_dfmp2_pwCV.pdf", note="XG_at_MAF5")

In [ ]:
ref_filters = dict(
    bases = [bas for bas in ref_data.bases if 'cc-pw' in bas],
    #core = ['core,0'],
)

filters = dict(
    #ansatzes = ['3C(FIX,HY1)'],
    bases = [bas for bas in data.bases if 'cc-pV' in bas],
    xg_maf_folders = ['MAF_CORE_5']
    #core = ['core,1'],
    #GEM_BETA = ['aug-cc-pVTZ'],
)
cmap = pp.get_cmap('viridis')

gamma_sets =  [
    '3.00_3.00_3.00',
    #'1.20_3.00_1.70', 
    '1.20_3.00_2.10', 
    #'1.20_3.00_2.50', 
    '1.20_3.00_3.00', 
]
diff2comp = [
    ('core,0', 'core,1'), 
    ('core,0', 'core,5'),
    ('core,1', 'core,5')
]



fig = pp.figure()
NGx = 3
NGy = 3
gs = pp.GridSpec(
    NGx, NGy, figure=fig,
    hspace=0.5
)

#colors = [cmap(i) for i in np.linspace(0, 0.8, n)]

fig.suptitle('Ar difference of correners, including BSE ')

axes = [[] for _ in range(NGx)]

sharex = None
for n in range(NGx):

    

    if n > 0:
        sharex = axes[n-1][m]

    sharey=None
    for m in range(NGy):
        if m > 0:
            sharey = axes[n][m - 1]
            
        ax = fig.add_subplot(
                gs[n * (NGx//3) : (n+1) * (NGx//3), m * (NGy//3) : (m+1) * (NGy//3) ] , sharex=sharex, sharey=sharey
            )
        axes[n].append(ax)

        if m > 0:
            pp.setp(axes[n][m].get_yticklabels(), visible = False)
        if n < NGx - 1:    
            pp.setp(axes[n][m].get_xticklabels(), visible=False)

        #for i, gamma in enumerate(gammas):
        
        filters['gamma_set'] = [gamma_sets[m]]
        
        filters['core'] = [diff2comp[n][0]]
        ref_filters['core'] = [diff2comp[n][0]]
        df0 = data.copy()
        for key, values in filters.items():
            df0 = df0[df0[key].isin(values)]
        #====================
        rdf0 = ref_data.copy()
        for key, values in ref_filters.items():
            rdf0 = rdf0[rdf0[key].isin(values)]

        filters['core'] = [diff2comp[n][1]]
        ref_filters['core'] = [diff2comp[n][1]]
        df1 = data.copy()
        for key, values in filters.items():
            df1 = df1[df1[key].isin(values)]
        #====================
        rdf1 = ref_data.copy()
        for key, values in ref_filters.items():
            rdf1 = rdf1[rdf1[key].isin(values)]

        
        #display(df)
        ener_diff = df0['correlation energy'].values - df1['correlation energy'].values
        ref_ener_diff = rdf0.CORR_ENER.values - rdf1.CORR_ENER.values

        basis_sizes = df0.basis_sizes.values

        BSE_ener0 = get_cbs_from_pair(basis_sizes[-2:], df0['correlation energy'].values[-2:])
        BSE_ener1 = get_cbs_from_pair(basis_sizes[-2:], df1['correlation energy'].values[-2:])

        ref_BSE_ener0 = get_cbs_from_pair(basis_sizes[-2:], rdf0.CORR_ENER.values[-2:])
        ref_BSE_ener1 = get_cbs_from_pair(basis_sizes[-2:], rdf1.CORR_ENER.values[-2:])
        
        ener_diff = np.append(ener_diff, [BSE_ener0 - BSE_ener1])
        ref_ener_diff = np.append(ref_ener_diff, [ref_BSE_ener0 - ref_BSE_ener1])

        x = 1/basis_sizes**(exp)
        x = np.append(x, [0])
        #display(x, ref_ener_diff)
        ax.plot(x, ref_ener_diff, marker = 'x', color=ref_colors[1])
        ax.plot(x, ener_diff, label=f'{gamma}', marker='o', color=colors[1])

for n in range(NGx):
    axes[0][n].set_title(f"{gamma_sets[n]}")

for n in range(NGy):
    axes[n][0].set_ylabel(f"{diff2comp[n]}")


In [ ]:
dr.dump_daily(fig, fname='XG_Ar_bse_cor_ener.pdf', note='xg runs, exp=3')

## [OLD]Results: BSE DFMP2 references

### Ne

In [ ]:
system = "test_frozen/"
method = "dfmp2"
data = get_data(f"{system}/{method}/data.csv")
assign_basis_size(data)
df0 = data[data.core == 'core,0']
df1 = data[data.core == 'core,1']

method = "default"
datf12 =  get_data(f"{system}/{method}/data.csv")
datf12 = datf12.fillna(0)

assign_basis_size(datf12)

In [ ]:
len([0,1,5,6,8,9,10,20,21,22,23,24,25,39,40,41,42,43,44,45,46,47,48,61,62,63,64,65,66,67])

In [ ]:
df_c0 = datf12[datf12.core=='core']
df_c1 = datf12[datf12.core=='core,1']

basis = 'aug-cc-pwCVDZ'
filters = {
    'ansatzes' : ['3C(FIX,NOZ)'],
    'bases' : [basis]
}

for key, vals in filters.items():
    df_c0 = df_c0[df_c0[key].isin(vals)]
    df_c1 = df_c1[df_c1[key].isin(vals)]
    
ax = df_c0.plot(x='GEM_BETA', y='CORR_ENER', label='core,0')
df_c1.plot(x='GEM_BETA', y='CORR_ENER', ax=ax)
print(set(datf12.bases.values))
ax.set_title(basis)

In [ ]:
help(dump_daily)
fig = ax.get_figure()
note = '''
Here we see that we don't have the same crossing of
trend lines as in the the case of aug-cc-pVDZ
'''
dump_daily(fig, fname=f'{basis}_corener_vs_gamma.pdf', note=note)

In [ ]:
data.bases

In [ ]:

line, ax = plot_cbs_extrapolation(
    df0.basis_sizes.values, df0.CORR_ENER.values,
    label = 'dfmp2,core,0', marker = 'o', color='tab:orange'
)

line, ax = plot_cbs_extrapolation(
    df1.basis_sizes.values, df1.CORR_ENER.values, ax = ax,
    label = 'dfmp2,core,1', marker='+', color='tab:orange'
)


dff12 = datf12.copy()

gamma = 1.4
linestyles = ['solid', 'dotted', 'dashdot']
cmap = pp.get_cmap('Blues_r')

n = len(linestyles)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 0.8, n)]

basis_type = 'aug-cc-pwCV'
ax.set_title(f'Ne, frozen vs unfrozen, {basis_type}')
for n, gamma in enumerate([1.0, 1.4, 2.0]):

    filters = {
        'GEM_BETA' : [gamma],
        'ansatzes' : ['3C(FIX,NOZ)'],
        'core' : ['core'],
        'bases' : [bas for bas in dff12.bases if basis_type in bas]
    }
    
    
    dff12 = datf12.copy()
    for key, val in filters.items():
        dff12 = dff12[dff12[key].isin(val)]
        
    #display(dff12)
    plot_cbs_extrapolation(dff12.basis_sizes.values, dff12.CORR_ENER.values,
                           ax = ax, label=f'df-mp2-f12,core,0;gamma={gamma}', marker = 'o',
                           color=colors[n], linestyle=linestyles[n]
                          )
    
    filters['core'] = ['core,1']
    
    dff12 = datf12.copy()
    for key, val in filters.items():
        dff12 = dff12[dff12[key].isin(val)]
        
    #display(dff12)
    plot_cbs_extrapolation(dff12.basis_sizes.values, dff12.CORR_ENER.values,
                           ax = ax, label=f'df-mp2-f12,core,1;gamma={gamma}', marker='+',
                           color=colors[n], linestyle=linestyles[n]
                          )
    


ax.set_xlabel('Basis size $X^{-3}$')
ax.set_ylabel('Correlation energy')

ax.legend()

In [ ]:
fig = ax.get_figure()

dump_daily(fig, fname=f'cbs_for_ne_diff_methods_{basis_type}', note="aug-cc-pwCVXZ for both dfmp2 and f12")

### Ar 

In [ ]:
system = "ar_frozen/"
method = "dfmp2"
data = get_data(f"{system}/{method}/data.csv")
ar_dfmp2_reference_data = data
assign_basis_size(data)
df0 = data[data.core == 'core,0']
df1 = data[data.core == 'core,1']
df5 = data[data.core == 'core,5']

#### DFMP2 BSE at different core specifications

In [ ]:
fig = pp.figure()
gs = pp.GridSpec(1, 7)

ax = fig.add_subplot(gs[:, 0:3])
axr =fig.add_subplot(gs[:, 3:]) 

line, ax = plot_cbs_extrapolation(
    df0.basis_sizes.values, df0.CORR_ENER.values, ax = ax,
    label = 'dfmp2,core,0', marker = 'o', color='tab:orange'
)
ax.lines
ax.set_title('Ar correlation energy')
line, ax = plot_cbs_extrapolation(
    df1.basis_sizes.values, df1.CORR_ENER.values, ax = ax,
    label = 'dfmp2,core,1', marker='+', color='tab:orange'
)
line, ax = plot_cbs_extrapolation(
    df5.basis_sizes.values, df5.CORR_ENER.values, ax = ax,
    label = 'dfmp2,core,5', marker='s', color='tab:orange'
)

In [ ]:
df0

#### F12 inspecting a single basis vs gamma

In [ ]:
method = "default"

datf12 =  get_data(f"{system}/{method}/data.csv")
assign_basis_size(datf12)


#display(df_c0)
basis = 'aug-cc-pwCVTZ'
filters = {
    'ansatzes' : ['3C(FIX,NOZ)'],
    'bases' : [basis]
}


filters['core'] = ['core,0']
dff120 = datf12.copy()
for key, vals in filters.items():
    dff120 = dff120[dff120[key].isin(vals)]

filters['core'] = ['core,1']
dff121 = datf12.copy()
for key, vals in filters.items():
    dff121 = dff121[dff121[key].isin(vals)]

filters['core'] = ['core,5']
dff125 = datf12.copy()
for key, vals in filters.items():
    dff125 = dff125[dff125[key].isin(vals)]


ax = dff120.plot(x='GEM_BETA', y='CORR_ENER', label='core,0')
dff121.plot(x='GEM_BETA', y='CORR_ENER', label= ' core,1', ax=ax)
dff125.plot(x='GEM_BETA', y='CORR_ENER', label= ' core,5', ax=ax)

ax.set_title(basis)

#### Comparisons to F12

In [ ]:
fig = pp.figure()
gs = pp.GridSpec(1, 7)

ax = fig.add_subplot(gs[:, 0:3])
axl =fig.add_subplot(gs[:, 3:]) 

line, ax = plot_cbs_extrapolation(
    df0.basis_sizes.values, df0.CORR_ENER.values, ax = ax,
    label = 'dfmp2,core,0', marker = 'o', color='tab:orange'
)
ax.set_title('Ar, frozen vs unfrozen')
line, ax = plot_cbs_extrapolation(
    df1.basis_sizes.values, df1.CORR_ENER.values, ax = ax,
    label = 'dfmp2,core,1', marker='+', color='tab:orange'
)
line, ax = plot_cbs_extrapolation(
    df5.basis_sizes.values, df5.CORR_ENER.values, ax = ax,
    label = 'dfmp2,core,5', marker='s', color='tab:orange'
)

gamma = 1.5
linestyles = ['solid', 'dotted', 'dashdot']
cmap = pp.get_cmap('Blues_r')

n = len(linestyles)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 0.8, n)]

for n, gamma in enumerate([1, 1.5, 2.0]):
    
    filters = {
        'GEM_BETA' : gamma,
        'ansatzes' : '3C(FIX,NOZ)',
        'core' : 'core,0'
    }
    
    dff12 = datf12.copy()
    for key, val in filters.items():
        dff12 = dff12[dff12[key] == val]
        
    display(dff12)
    plot_cbs_extrapolation(dff12.basis_sizes.values, dff12.CORR_ENER.values,
                           ax = ax, label=f'df-mp2-f12,core,0;gamma={gamma}', marker='o', color=colors[n]
                          )
    
    filters['core'] = 'core,1'
    dff12 = datf12.copy()
    for key, val in filters.items():
        dff12 = dff12[dff12[key] == val]
        
    #display(dff12)
    plot_cbs_extrapolation(dff12.basis_sizes.values, dff12.CORR_ENER.values,
                           ax = ax, label=f'df-mp2-f12,core,1;gamma={gamma}', marker='+', color=colors[n]
                          )
    
    datf12 = datf12.fillna(0)
    
    filters['core'] = 'core,5'
    
    dff12 = datf12.copy()
    for key, val in filters.items():
        dff12 = dff12[dff12[key] == val]
        
    #display(dff12)
    plot_cbs_extrapolation(dff12.basis_sizes.values, dff12.CORR_ENER.values,
                           ax = ax, label=f'df-mp2-f12,core,5;gamma={gamma}', marker='s', color=colors[n]
                          )

axl.set_axis_off()
axl.legend(*ax.get_legend_handles_labels())

In [ ]:
note = '''
same kind of trend as Ne, but over more sets of frozen orbitals 
- at DZ, the gamma is sensitive to which orbitals are frozen
'''
dump_daily(fig, fname='cbs_for_ar_different_methods', note=note)

## Results: Interaction energies 

### Ne-Ar

#### Interaction Energy for all Gammas at a particular basis set

In [ ]:
system = "ne_ar"
method0 = "default"
data0 = get_data(f"{system}/{method0}/data.csv", ignore_file_cols=False)
assign_basis_size(data0)
#bases = set([bas for bas in data0.bases if 'pV' in bas])
#data0.fillna(value={'core':'core,1'}, inplace=True)
gammas = list(set(data0.GEM_BETA.values))


fig, ax = pp.subplots()
cmap = pp.get_cmap('viridis')
n = len(gammas)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 1, n)]
fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(gammas[0], gammas[-1]), cmap=cmap),
             ax=ax, orientation='vertical', label='gamma')


for n, gamma in enumerate(gammas):
    filters = dict(
        ansatzes = '3C(FIX,HY1)',
        core = 'core,0',
        bases = 'aug-cc-pVTZ',
        GEM_BETA = gamma,
    )
    df = data0
    for key, val in filters.items():
        df = df[df[key] == val]
    inf_ener = df.TOT_ENER.values[-1]
    ax.plot(
        df.distances.values, df.TOT_ENER.values - inf_ener, marker='x',
        color=colors[n], alpha=0.5
    )

ax.set_title(f"Interaction energy of {system} at {filters['bases']}")
#ax.legend()
ax.set_xlim(3.2, 4.0)
ax.set_ylim(-0.00025,0)
ax.set_xlabel(r'Distance, r ($\AA$)')
ax.set_ylabel(r'Interaction energy (hartree)')
fig.tight_layout()
df
#print(df.distances)
#ax.set_ylim(-656, -655.85)    

In [ ]:
system = "ne_ar"
method0 = "default"
data0 = get_data(f"{system}/{method0}/data.csv", ignore_file_cols=False)
assign_basis_size(data0)
#bases = set([bas for bas in data0.bases if 'pV' in bas])
data0.fillna(value={'core':'core,1'}, inplace=True)
gammas = list(set(data0.GEM_BETA.values))


fig, ax = pp.subplots()
cmap = pp.get_cmap('viridis')
n = len(gammas)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 1, n)]
fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(gammas[0], gammas[-1]), cmap=cmap),
             ax=ax, orientation='vertical', label='gamma')


for n, gamma in enumerate(gammas):
    filters = dict(
        ansatzes = '3C(FIX,HY1)',
        core = 'core,1',
        bases = 'aug-cc-pVTZ',
        GEM_BETA = gamma,
    )
    df = data0
    for key, val in filters.items():
        df = df[df[key] == val]
    inf_ener = df.TOT_ENER.values[-1]
    ax.plot(
        df.distances.values, df.TOT_ENER.values - inf_ener, marker='x',
        color=colors[n], alpha=0.5
    )

ax.set_title(f"Interaction energy of {system} at {filters['bases']}")
#ax.legend()
ax.set_xlim(3.2, 4.0)
ax.set_ylim(-0.00025,0)
ax.set_xlabel(r'Distance, r ($\AA$)')
ax.set_ylabel(r'Interaction energy (hartree)')
fig.tight_layout()
df
#print(df.distances)
#ax.set_ylim(-656, -655.85)    

In [ ]:
dr.dump_daily(fig, note='core,1')

#### Basis set extrapolation at minimum

In [ ]:
basis_size_dict = {}
for n, basis in enumerate(data0.bases.values):
    basis_size_dict[basis] = data0.basis_sizes.values[n]
    
bases = [bas for bas in set(data0['bases'].values) if 'cc-pV' in bas]
bases = sorted(bases, key=lambda bas: basis_size_dict[bas])

fig,ax = pp.subplots()
gammas = list(set(data0.GEM_BETA.values))[5:-5:2]
cmap = pp.get_cmap('viridis')
n = len(gammas)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 1, n)]
fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(gammas[0], gammas[-1]), cmap=cmap),
             ax=ax, orientation='vertical', label='gamma')

for n, gamma in enumerate(gammas):
    filters = dict(
        ansatzes = ['3C(FIX)'],
        core = ['core,1'],
        bases = bases,
        GEM_BETA = [gamma],
        distances = [3.5]
    )
    df = data0
    for key, val in filters.items():
        df = df[df[key].isin(val)]
    plot_cbs_extrapolation(
        df.basis_sizes.values, df.CORR_ENER.values,
        ax = ax, label=f'gamma={gamma}', marker='o', color=colors[n],
        exp=3
    )

ax.set_title(f"BSE of {system} at various gamma")

In [ ]:
dr.dump_daily(fig, fname='BSE_ne_ar_default_core1.pdf', note='core,1')

# Optimal gamma

In [ ]:
basis_size_dict = {}
for n, basis in enumerate(data0.bases.values):
    basis_size_dict[basis] = data0.basis_sizes.values[n]
    
bases = [bas for bas in set(data0['bases'].values) if 'cc-pV' in bas]
bases = sorted(bases, key=lambda bas: basis_size_dict[bas])

fig,ax = pp.subplots()
gammas = list(set(data0.GEM_BETA.values))[5:-5:2]



filters = dict(
    ansatzes = ['3C(FIX)'],
    core = ['core,1'],
    bases = ['aug-cc-pVTZ'],
    distances = [3.5]
)
df = data0
for key, val in filters.items():
    df = df[df[key].isin(val)]
    
ax.plot(df.GEM_BETA.values, df.CORR_ENER.values)
ax.axvline(
    get_x_at_ymin(df.GEM_BETA.values, df.CORR_ENER.values), color='k', linestyle='--'
)
    
ax.set_title(f"Corr E vs Gamma of {system} at r={filters['distances']}")

#### XG

In [ ]:
system = "ne_ar"
method0 = "xg"
data0 = get_data(f"{system}/{method0}/data.csv")
assign_basis_size(data0)

set(data0.gamma_set)
display(data0.columns)

#bases = set([bas for bas in data0.bases if 'pV' in bas])
#gamma_sets = list(set(data0.GEM_BETA.values))


fig, ax = pp.subplots()
cmap = pp.get_cmap('viridis')
gamma_sets = list(set(data0.gamma_set))
n = len(gamma_sets)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 0.95, n)]
#fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(gamma_sets[0], gamma_sets[-1]), cmap=cmap),
#             ax=ax, orientation='vertical', label='gamma')


for n, gamma in enumerate(gamma_sets):
    filters = dict(
        #ansatzes = '3C(FIX)',
        core_names = 'default_core',
        bases = 'aug-cc-pVTZ',
        gamma_set = gamma,
    )
    df = data0
    for key, val in filters.items():
        df = df[df[key] == val]
    inf_ener = df['total energy'].values[-1]
    int_e = df['total energy'].values - inf_ener
    ax.plot(
        df.distances.values, int_e, marker='x',
        color=colors[n], alpha=0.5, label = gamma
    )
    display(int_e)
ax.set_title(f"Interaction energy of {system} at {filters['bases']}")
#ax.legend()
ax.set_xlim(3.2, 4.0)
ax.set_ylim(-0.00025,0)

ax.set_xlabel(r'Distance, r ($\AA$)')
ax.set_ylabel(r'Interaction energy (hartree)')
ax.legend()
fig.tight_layout()
#print(df.distances)
#ax.set_ylim(-656, -655.85)    a

In [ ]:
fig, ax = pp.subplots()
cmap = pp.get_cmap('viridis')
gamma_sets = list(set(data0.gamma_set))
n = len(gamma_sets)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 0.95, n)]
ax.set_title(f"BSE of {system} at various gamma sets (XG)")
for n, gamma in enumerate(gamma_sets):
    filters = dict(
        core_names = ['default_core'],
        bases = bases,
        gamma_set = [gamma],
        distances = [3.5]
    )
    df = data0
    for key, val in filters.items():
        df = df[df[key].isin(val)]
    plot_cbs_extrapolation(
        df.basis_sizes.values, df['correlation energy'].values,
        ax = ax, label=f'gamma={gamma}', marker='o', color=colors[n],
        exp=3
    )
ax.legend()

In [ ]:
get_data('cu_nh3/standard/data.csv').columns

### Cu_NH3

#### Default

In [ ]:
system = "cu_nh3"
method0 = "default"
data0 = get_data(f"{system}/{method0}/data.csv")
assign_basis_size(data0)
#bases = set([bas for bas in data0.bases if 'pV' in bas])
gammas = list(set(data0.GEM_BETA.values))
data0.columns

In [ ]:
fig, ax = pp.subplots()
cmap = pp.get_cmap('viridis')
n = len(gammas)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 1, n)]
fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(gammas[0], gammas[-1]), cmap=cmap),
             ax=ax, orientation='vertical', label='gamma')


for n, gamma in enumerate(gammas):
    filters = dict(
        ansatzes = '3C(FIX,HY1)',
        #core = 'core,0',
        bases = 'aug-cc-pVTZ',
        GEM_BETA = gamma,
    )
    df = data0
    for key, val in filters.items():
        df = df[df[key] == val]
    inf_ener = df.TOT_ENER.values[-1]
    ax.plot(
        df.distances.values, df.TOT_ENER.values - inf_ener, marker='x',
        color=colors[n], alpha=0.5
    )

ax.set_title(f"Interaction energy of {system} at {filters['bases']}")
#ax.legend()
ax.set_xlim(3.2, 4.0)
ax.set_ylim(-0.00025,0)
ax.set_xlabel(r'Distance, r ($\AA$)')
ax.set_ylabel(r'Interaction energy (hartree)')
fig.tight_layout()
df

#### XG

In [ ]:
system = "cu_nh3"
method0 = "xg"
data0 = get_data(f"{system}/{method0}/data.csv")
assign_basis_size(data0)

set(data0.gamma_set)
display(data0.columns)
display(set(data0.core_names))

#bases = set([bas for bas in data0.bases if 'pV' in bas])
#gamma_sets = list(set(data0.GEM_BETA.values))


fig, ax = pp.subplots()
cmap = pp.get_cmap('viridis')
gamma_sets = list(set(data0.gamma_set))
n = len(gamma_sets)  # number of colors needed
colors = [cmap(i) for i in np.linspace(0, 0.95, n)]
#fig.colorbar(mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(gamma_sets[0], gamma_sets[-1]), cmap=cmap),
#             ax=ax, orientation='vertical', label='gamma')


for n, gamma in enumerate(gamma_sets):
    filters = dict(
        #ansatzes = '3C(FIX)',
        core_names = 'default_core',
        bases = 'aug-cc-pVTZ',
        gamma_set = gamma,
    )
    df = data0
    for key, val in filters.items():
        df = df[df[key] == val]
    inf_ener = df['total energy'].values[-1]
    int_e = df['total energy'].values - inf_ener
    ax.plot(
        df.distances.values, int_e, marker='x',
        color=colors[n], alpha=0.5, label = gamma
    )
    display(int_e)
ax.set_title(f"Interaction energy of {system} at {filters['bases']} (XG)")
#ax.legend()
ax.set_xlim(1.8, 3.0)
ax.set_ylim(-0.1,0)

ax.set_xlabel(r'Distance, r ($\AA$)')
ax.set_ylabel(r'Interaction energy (hartree)')
ax.legend()
fig.tight_layout()